# Attention multimodal alignment
Run the numbered blocks in order. Source folders are read-only; export is disabled by default.

In [ ]:
# Block 1 - environment, paths, and session inventory
%matplotlib widget
from pathlib import Path
import json
import matplotlib.pyplot as plt
import pandas as pd
from attention_alignment import load_config
from attention_alignment.pipeline import build_alignment, export_session

CONFIG_PATH = Path('configs/sessions.local.yaml')
CONFIG = load_config(CONFIG_PATH)
assert CONFIG.data_root.is_dir() and CONFIG.stimuli_root.is_dir()
assert CONFIG.project_root in CONFIG.output_root.parents
pd.DataFrame([vars(item) for item in CONFIG.sessions])

In [ ]:
# Block 2 - strict order parsing and 200 x 5 design
from attention_alignment.stimulus import parse_order_file, sequence_table

designs = {}
for label in (500, 1000):
    order = parse_order_file(CONFIG.order_path(label))
    designs[label] = sequence_table(order, CONFIG.stimulus['angle_degrees'])
    assert len(designs[label]) == 1000
    print(label, designs[label].query("phase == 'test' and item_position == 5")['symbol'].value_counts().to_dict())
designs[500].head(10)

In [ ]:
# Block 3 - reference video frame timing and expected visual runs
from attention_alignment.video import reference_video_runs

reference_runs = {label: reference_video_runs(CONFIG.reference_video_path(label)) for label in (500, 1000)}
display(reference_runs[500].head(11))
display(reference_runs[1000].head(11))

In [ ]:
# Block 4 - TXT/DAT segmentation and main recording selection (read-only)
RESULTS = build_alignment(CONFIG, dry_run=True)
summary = pd.DataFrame([result.summary() for result in RESULTS.values()])
display(summary)
assert len(RESULTS) == 8

In [ ]:
# Block 5 - first-calcium 01/02 synchronization residuals
alignment_qc = pd.DataFrame([{'session_id': sid, **r.qc['marker_alignment']} for sid, r in RESULTS.items()])
display(alignment_qc)
ax = alignment_qc.set_index('session_id')[['calcium_max_abs_residual_s', 'trigger_max_abs_nearest_residual_s']].plot.bar(figsize=(12, 4))
ax.set_ylabel('maximum absolute residual (s)')
plt.tight_layout()

In [ ]:
# Block 6 - prelude classification, Train/Test templates, and event preview
SESSION_ID = next(iter(RESULTS))
result = RESULTS[SESSION_ID]
display(pd.DataFrame(result.qc['triggers'].items(), columns=['metric', 'value']))
display(result.stimulus_events.head(12))
display(result.stimulus_events.groupby(['phase', 'symbol']).size())

In [ ]:
# Block 7 - calcium frame binding and tail-pulse audit
calcium_qc = pd.DataFrame([{'session_id': sid, **r.qc['calcium']} for sid, r in RESULTS.items()])
display(calcium_qc)
display(result.calcium_frames.head())

In [ ]:
SESSION_ID

In [ ]:
# Block 8 - per-session/camera ROI review
from dataclasses import asdict
from attention_alignment.behavior import (
    NotebookROISelector, ensure_session_roi_config, load_roi_config,
    preview_pupil_detection, save_roi_config,
)

# SESSION_ID = "replace_with_session_id"
CAMERA = '01'
FRAME_INDEX = 100
video_path = CONFIG.video_path(SESSION_ID, CAMERA)
eye_selector = NotebookROISelector(video_path, FRAME_INDEX, f'{SESSION_ID} {CAMERA} eye ROI')
# preview_pupil_detection(video_path, eye_selector.roi, [0, 300, 600])
# After drawing the eye ROI, create a movement selector in a new cell if desired:
# movement_selector = NotebookROISelector(video_path, FRAME_INDEX, f'{SESSION_ID} {CAMERA} movement ROI')
# preview_pupil_detection(video_path, eye_selector.roi, [0, 300, 600])
# roi_path = ensure_session_roi_config(
#     CONFIG.session_roi_config_path(SESSION_ID), SESSION_ID,
#     CONFIG.aggregate_roi_config_path(),
# )
# rois = load_roi_config(roi_path)
# rois.setdefault(SESSION_ID, {})[CAMERA] = {'eye': asdict(eye_selector.roi), 'movement': asdict(movement_selector.roi)}
# save_roi_config(roi_path, rois)

In [ ]:
preview_pupil_detection(video_path, eye_selector.roi, [0, 300, 600])

In [ ]:
# After drawing the eye ROI, create a movement selector in a new cell if desired:
CAMERA = '02'
FRAME_INDEX = 100
video_path = CONFIG.video_path(SESSION_ID, CAMERA)
movement_selector = NotebookROISelector(video_path, FRAME_INDEX, f'{SESSION_ID} {CAMERA} movement ROI')

In [ ]:
preview_pupil_detection(video_path, eye_selector.roi, [0, 300, 600])

In [ ]:
roi_path = ensure_session_roi_config(
    CONFIG.session_roi_config_path(SESSION_ID), SESSION_ID,
    CONFIG.aggregate_roi_config_path(),
)
rois = load_roi_config(roi_path)
rois.setdefault(SESSION_ID, {})[CAMERA] = {'eye': asdict(eye_selector.roi), 'movement': asdict(movement_selector.roi)}
save_roi_config(roi_path, rois)

In [ ]:
# Block 9 - behavior batch extraction (explicit opt-in)
RUN_BEHAVIOR = False
if RUN_BEHAVIOR:
    from attention_alignment.pipeline import export_session
    export_session(CONFIG, result, include_video_indexes=False, include_behavior=True)
else:
    print('Behavior extraction skipped; review and save both ROIs first.')

In [ ]:
# Block 10 - lazy 02 electrophysiology window
from attention_alignment.ephys import read_ephys_window

onset = float(result.stimulus_events.iloc[0]['measured_onset_s'])
ephys_window = read_ephys_window(result.manifest['sources']['ephys_02'], onset - 1, onset + 3)
display(ephys_window.head())
ephys_window.plot(x='t_session_s', y=['EEG1', 'EEG2', 'EMG'], subplots=True, figsize=(10, 6))

In [ ]:
# Block 11 - random trial cross-modal timing overlay
trial = result.stimulus_events.sample(1, random_state=7).iloc[0]
t0 = float(trial['measured_onset_s'])
window = (-2.0, 5.0)
calcium_index = result.calcium_frames.loc[(result.calcium_frames['t_session_s'] >= t0 + window[0]) & (result.calcium_frames['t_session_s'] < t0 + window[1])].copy()
ephys_index = read_ephys_window(result.manifest['sources']['ephys_02'], t0 + window[0], t0 + window[1], ['EEG1'])
fig, axes = plt.subplots(2, 1, sharex=True, figsize=(10, 6))
axes[0].plot(calcium_index['t_session_s'] - t0, calcium_index['frame_index'], '.-')
axes[0].axvspan(0, float(trial['measured_duration_s']), alpha=.2, color='tab:orange')
axes[0].set_ylabel('calcium frame')
axes[1].plot(ephys_index['t_session_s'] - t0, ephys_index['EEG1'], lw=.5)
axes[1].set(xlabel='time from measured onset (s)', ylabel='EEG1')
plt.tight_layout()

In [ ]:
# Block 12 - reviewed export (explicit opt-in)
EXPORT = True
if EXPORT:
    exported = {sid: export_session(CONFIG, item) for sid, item in RESULTS.items()}
    display(exported)
else:
    print('Dry-run complete. Set EXPORT=True only after reviewing Blocks 1-11.')

In [ ]:
SESSION_ID = "replace_with_session_id"
result = RESULTS[SESSION_ID]

display(result.summary())
display(result.calcium_frames.head())
display(result.stimulus_events.head())

export_session(CONFIG, result)